# Build reproducible samples

In [ ]:
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

DATA_ROOT = Path("S:/Zulassungskarten_Data")
SOURCE_DIR = DATA_ROOT / "R_9346-I_Zulassungskarten"
SAMPLE_DIR = DATA_ROOT / "sample_dataset"
JSON_DIR = DATA_ROOT / "sample_json"
GROUND_TRUTH_DIR = DATA_ROOT / "ground_truth_dataset"

RANDOM_STATE = 42
ROWS_PER_MONTH = 10
GROUND_TRUTH_YEARS = 25 # extract 1 doc per year!! (maybe also a variable instead of hard coded??)

## Load and prepare data

In [ ]:
def load_data(base_dir: Path) -> pd.DataFrame:
    if not base_dir.is_dir():
        raise FileNotFoundError(f"Source directory does not exist: {base_dir}")

    records = []
    for json_path in sorted(base_dir.rglob("*.json")):
        try:
            with json_path.open("r", encoding="utf-8") as file:
                record = json.load(file)

            if not isinstance(record, dict):
                raise ValueError("JSON root is not an object")

            record["_source_folder"] = json_path.parent.name
            record["_file_name"] = json_path.name
            records.append(record)

        except (json.JSONDecodeError, OSError, ValueError) as error:
            print(f"Could not read {json_path}: {error}")

    return pd.DataFrame(records)


def prepare_data(data: pd.DataFrame) -> pd.DataFrame:
    prepared = data.copy()
    prepared["Prüfdatum"] = pd.to_datetime(prepared["Prüfdatum"], dayfirst=True, errors="coerce")
    prepared["Jahr"] = prepared["Prüfdatum"].dt.year.astype("Int64")
    prepared["Prüfnummer"] = pd.to_numeric(prepared["Prüfnummer"], errors="coerce")
    
    return prepared


In [ ]:
df = prepare_data(load_data(SOURCE_DIR))
print(f"Loaded {len(df):,} data records.")

## Sampling functions

In [ ]:
def sample_by_month(
    data: pd.DataFrame,
    rows_per_month: int = 10,
    random_state: int = 42,
    date_col: str = "Prüfdatum",
) -> pd.DataFrame:
    if rows_per_month < 1:
        raise ValueError("rows_per_month must be at least 1")
    if date_col not in data:
        raise KeyError(f"Missing date column: {date_col}")

    working = data.copy()
    working["_sample_date"] = pd.to_datetime(
        working[date_col], dayfirst=True, errors="coerce"
    )
    working = working.loc[working["_sample_date"].notna()].copy()
    working["_sample_year"] = working["_sample_date"].dt.year
    working["_sample_month"] = working["_sample_date"].dt.month
    working["_sample_order"] = np.random.default_rng(random_state).random(
        len(working)
    )

    sampled = (
        working.sort_values(
            ["_sample_year", "_sample_month", "_sample_order"]
        )
        .groupby(["_sample_year", "_sample_month"], sort=True)
        .head(rows_per_month)
        .sort_values(["_sample_year", "_sample_month", "_sample_order"])
        .drop(columns=["_sample_date", "_sample_year", "_sample_month", "_sample_order"])
        .reset_index(drop=True)
    )
    return sampled

# 1 sample per year
def sample_ground_truth(
    data: pd.DataFrame,
    number_of_years: int = 25,
    random_state: int = 42,
    date_col: str = "Prüfdatum",
) -> pd.DataFrame:
    if number_of_years < 1:
        raise ValueError("number_of_years must be at least 1")
    if date_col not in data:
        raise KeyError(f"Missing date column: {date_col}")

    working = data.copy()
    working["_sample_date"] = pd.to_datetime(
        working[date_col], dayfirst=True, errors="coerce"
    )
    working = working.loc[working["_sample_date"].notna()].copy()
    working["_sample_year"] = working["_sample_date"].dt.year.astype(int)
    available_years = sorted(working["_sample_year"].unique())

    if len(available_years) < number_of_years:
        raise ValueError(
            f"Only {len(available_years)} years are available; "
            f"cannot sample {number_of_years} years."
        )

    positions = np.rint(
        np.linspace(0, len(available_years) - 1, number_of_years)
    ).astype(int)
    selected_years = [available_years[position] for position in positions]
    candidates = working[working["_sample_year"].isin(selected_years)].copy()
    candidates["_sample_order"] = np.random.default_rng(random_state).random(
        len(candidates)
    )

    sampled = (
        candidates.sort_values(["_sample_year", "_sample_order"])
        .groupby("_sample_year", sort=True)
        .head(1)
        .sort_values("_sample_year")
        .drop(columns=["_sample_date", "_sample_year", "_sample_order"])
        .reset_index(drop=True)
    )

    if len(sampled) != number_of_years:
        raise RuntimeError("Could not select one record for every requested year")
    return sampled


## Create both samples

In [ ]:
sample_df = sample_by_month(
    df, rows_per_month=ROWS_PER_MONTH, random_state=RANDOM_STATE
)
print(f"Sample: {len(sample_df):,} records")
sample_df.head()

In [ ]:
ground_truth_df = sample_ground_truth(
    df, number_of_years=GROUND_TRUTH_YEARS, random_state=RANDOM_STATE
)
print(f"Ground truth: {len(ground_truth_df):,} records")
ground_truth_df

## Copy source folders

In [ ]:
def copy_folders(
    data: pd.DataFrame,
    source_dir: Path,
    target_dir: Path,
    folder_col: str = "_source_folder",
    overwrite: bool = False,
) -> dict[str, int]:
    if not source_dir.is_dir():
        raise FileNotFoundError(f"Source directory does not exist: {source_dir}")
    if folder_col not in data:
        raise KeyError(f"Missing folder column: {folder_col}")

    target_dir.mkdir(parents=True, exist_ok=True)
    summary = {"copied": 0, "skipped": 0, "missing": 0}

    folders = sorted(data[folder_col].dropna().astype(str).unique())
    for folder in folders:
        source_folder = source_dir / folder
        target_folder = target_dir / folder

        if not source_folder.is_dir():
            summary["missing"] += 1
            print(f"Source folder not found: {source_folder}")
            continue
        if target_folder.exists() and not overwrite:
            summary["skipped"] += 1
            continue

        shutil.copytree(source_folder, target_folder, dirs_exist_ok=overwrite)
        summary["copied"] += 1

    return summary


In [ ]:
def copy_json_files(
    data: pd.DataFrame,
    source_dir: Path,
    target_dir: Path,
    folder_col: str = "_source_folder",
    file_col: str = "_file_name",
    overwrite: bool = False,
) -> dict[str, int]:
    if not source_dir.is_dir():
        raise FileNotFoundError(f"Source directory does not exist: {source_dir}")
    if folder_col not in data or file_col not in data:
        raise KeyError(f"Missing columns: {folder_col}, {file_col}")

    target_dir.mkdir(parents=True, exist_ok=True)
    summary = {"copied": 0, "skipped": 0, "missing": 0}

    files = data[[folder_col, file_col]].dropna().drop_duplicates()
    for folder, file_name in files.itertuples(index=False):
        source_file = source_dir / str(folder) / str(file_name)
        target_file = target_dir / str(file_name)

        if not source_file.is_file():
            summary["missing"] += 1
            print(f"Source file not found: {source_file}")
            continue
        if target_file.exists() and not overwrite:
            summary["skipped"] += 1
            continue

        shutil.copy2(source_file, target_file)
        summary["copied"] += 1

    return summary


In [ ]:
sample_copy_summary = copy_folders(sample_df, SOURCE_DIR, SAMPLE_DIR)
sample_copy_summary

In [ ]:
sample_files_df = prepare_data(load_data(SAMPLE_DIR)) ## ich will die jsons aus dem sample (was schon gezogen wurde) kopieren 

sample_json_copy_summary = copy_json_files(sample_files_df, SOURCE_DIR, JSON_DIR)
sample_json_copy_summary

In [ ]:
ground_truth_copy_summary = copy_folders(ground_truth_df, SOURCE_DIR, GROUND_TRUTH_DIR)
ground_truth_copy_summary